# PrivAware v2 — Day 3: Push Model to HuggingFace + Deploy API

**Goal:** By end of today your phishing model is live at a public URL and your FastAPI backend is deployed on HuggingFace Spaces. No localhost anywhere.

**This notebook has 2 parts:**
- Part A (Cells 1–5): Push your trained model to HuggingFace Hub
- Part B (Cells 6–10): Create and deploy the FastAPI backend as a HuggingFace Space

**Before you start:**
1. Go to huggingface.co → Settings → Access Tokens → New Token → Role: **Write** → copy the token
2. Have your Google Drive mounted with the phishing_model folder from Day 2
3. Runtime → Change runtime type → T4 GPU → Save

---
# PART A — Push Model to HuggingFace Hub

## Cell 1 — Install and login to HuggingFace

In [ ]:
!pip install huggingface_hub transformers -q

from huggingface_hub import notebook_login

print('A box will appear below. Paste your HuggingFace WRITE token into it.')
print('Get it from: huggingface.co -> Settings -> Access Tokens -> New Token (Write)')
notebook_login()

## Cell 2 — Mount Drive and load your trained model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

DRIVE_MODEL_PATH = '/content/drive/MyDrive/privaware-v2/phishing_model/'

model     = DistilBertForSequenceClassification.from_pretrained(DRIVE_MODEL_PATH)
tokenizer = DistilBertTokenizerFast.from_pretrained(DRIVE_MODEL_PATH)

print('Model loaded from Drive!')
print(f'Labels: {model.config.num_labels} (0=legitimate, 1=phishing)')

## Cell 3 — Add model card info (important for panel)
This writes metadata about your model — what it does, what dataset, what accuracy.

In [ ]:
# Update YOUR_USERNAME and YOUR_ACCURACY below before running
YOUR_USERNAME = 'your-huggingface-username'   # e.g. v3d4nt7
YOUR_ACCURACY = '95.00'                        # paste your Day 2 test accuracy here
YOUR_F1       = '0.95'                         # paste your Day 2 F1 score here

model.config.id2label = {0: 'LEGITIMATE', 1: 'PHISHING'}
model.config.label2id = {'LEGITIMATE': 0, 'PHISHING': 1}
model.config.finetuning_task = 'text-classification'
model.config.problem_type = 'single_label_classification'

# Save updated config
model.save_pretrained('/content/best_model_final')
tokenizer.save_pretrained('/content/best_model_final')

# Write model card
model_card = f"""---
language: en
tags:
- text-classification
- phishing-detection
- cybersecurity
- distilbert
datasets:
- pirocheto/phishing-url
metrics:
- accuracy
- f1
---

# PrivAware Phishing URL Detector

A DistilBERT model fine-tuned to detect phishing URLs in real time.
Built as part of the PrivAware v2 Chrome extension project.

## Model Details
- **Base model:** distilbert-base-uncased
- **Task:** Binary text classification (phishing vs legitimate)
- **Dataset:** pirocheto/phishing-url (50,000 samples, balanced)
- **Test Accuracy:** {YOUR_ACCURACY}%
- **Test F1 Score:** {YOUR_F1}

## Labels
- `LEGITIMATE` (0) — safe URL
- `PHISHING` (1) — malicious/phishing URL

## Usage
```python
from transformers import pipeline
classifier = pipeline('text-classification', model='{YOUR_USERNAME}/privaware-phishing-detector')
result = classifier('http://totally-legit-bank.xyz/login')
print(result)
```
"""

with open('/content/best_model_final/README.md', 'w') as f:
    f.write(model_card)

print('Model card written!')
print(f'Username set to: {YOUR_USERNAME}')
print('If username is wrong, update it above and re-run this cell.')

## Cell 4 — Push model to HuggingFace Hub
This uploads your model files. Takes 3-5 minutes depending on internet speed.

In [ ]:
REPO_ID = f'{YOUR_USERNAME}/privaware-phishing-detector'

print(f'Pushing model to: https://huggingface.co/{REPO_ID}')
print('This takes 3-5 minutes...')

model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)

print('\nModel pushed successfully!')
print(f'View it at: https://huggingface.co/{REPO_ID}')

## Cell 5 — Verify: run a quick inference test
Make sure the model actually works from the Hub before moving to Part B.

In [ ]:
from transformers import pipeline
import time

print('Loading model from HuggingFace Hub to verify...')
classifier = pipeline('text-classification', model=REPO_ID)

test_urls = [
    'https://www.google.com',
    'https://github.com/login',
    'http://secure-paypal-login.xyz/verify?user=victim',
    'http://amazon-account-suspended.tk/restore',
    'https://stackoverflow.com/questions'
]

print('\nInference test results:')
print('-' * 60)
for url in test_urls:
    result = classifier(url)[0]
    label = result['label']
    score = result['score']
    flag = '🚨' if label == 'PHISHING' else '✅'
    print(f'{flag} {label} ({score*100:.1f}%) — {url[:55]}')

print('\nPart A complete! Model is live on HuggingFace.')
print(f'URL: https://huggingface.co/{REPO_ID}')

---
# PART B — Deploy FastAPI Backend as HuggingFace Space

A HuggingFace Space is a free hosted app. We deploy a FastAPI app there so the Chrome extension can call it from anywhere — no localhost.

**Before running Cell 6:**
1. Go to huggingface.co/new-space
2. Space name: `privaware-api`
3. SDK: **Docker**
4. Visibility: **Public**
5. Click Create Space
6. Copy the Space URL shown — it will be like: `https://huggingface.co/spaces/YOUR_USERNAME/privaware-api`

## Cell 6 — Write the FastAPI app file

In [ ]:
import os
os.makedirs('/content/space_files', exist_ok=True)

# Update YOUR_USERNAME below
YOUR_USERNAME = 'your-huggingface-username'   # same as above

app_code = f'''
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from transformers import pipeline
import uvicorn

app = FastAPI(title="PrivAware API", version="2.0")

# Allow Chrome extension to call this API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Load models once at startup
print("Loading phishing model...")
phishing_classifier = pipeline(
    "text-classification",
    model="{YOUR_USERNAME}/privaware-phishing-detector"
)
print("Phishing model loaded!")

class URLRequest(BaseModel):
    url: str

class PolicyRequest(BaseModel):
    text: str

@app.get("/")
def root():
    return {{"status": "ok", "message": "PrivAware API is running"}}

@app.get("/health")
def health():
    return {{"status": "healthy", "models": ["phishing-detector"]}}

@app.post("/scan-url")
def scan_url(request: URLRequest):
    if not request.url:
        raise HTTPException(status_code=400, detail="URL is required")
    try:
        result = phishing_classifier(request.url[:512])[0]
        label = result["label"]
        confidence = round(result["score"] * 100, 2)
        risk_score = round(confidence) if label == "PHISHING" else round(100 - confidence)
        return {{
            "url": request.url,
            "label": label,
            "confidence": confidence,
            "risk_score": risk_score,
            "is_phishing": label == "PHISHING"
        }}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/scan-policy")
def scan_policy(request: PolicyRequest):
    # Policy model comes Day 4 — for now returns a placeholder
    if not request.text:
        raise HTTPException(status_code=400, detail="Text is required")
    return {{
        "label": "PENDING",
        "message": "Privacy policy model coming Day 4",
        "risk_score": 50
    }}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=7860)
'''

with open('/content/space_files/app.py', 'w') as f:
    f.write(app_code)

print('app.py written!')
print('Preview:')
print(app_code[:500] + '...')

## Cell 7 — Write requirements.txt and Dockerfile

In [ ]:
requirements = """fastapi==0.104.1
uvicorn==0.24.0
transformers==4.35.2
torch==2.1.0
pydantic==2.4.2
httpx==0.25.1
"""

dockerfile = """FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

EXPOSE 7860

CMD ["python", "app.py"]
"""

with open('/content/space_files/requirements.txt', 'w') as f:
    f.write(requirements)

with open('/content/space_files/Dockerfile', 'w') as f:
    f.write(dockerfile)

print('requirements.txt written!')
print('Dockerfile written!')
print('\nFiles ready to upload:')
import os
for f in os.listdir('/content/space_files'):
    print(f'  {f}')

## Cell 8 — Upload all files to your HuggingFace Space

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
SPACE_REPO = f'{YOUR_USERNAME}/privaware-api'

print(f'Uploading files to Space: {SPACE_REPO}')

for filename in ['app.py', 'requirements.txt', 'Dockerfile']:
    api.upload_file(
        path_or_fileobj=f'/content/space_files/{filename}',
        path_in_repo=filename,
        repo_id=SPACE_REPO,
        repo_type='space'
    )
    print(f'  Uploaded: {filename}')

print('\nAll files uploaded!')
print('HuggingFace is now building your Space...')
print('This takes 3-7 minutes.')
print(f'\nWatch the build at:')
print(f'https://huggingface.co/spaces/{SPACE_REPO}')

## Cell 9 — Wait for Space to build then test it

In [ ]:
import requests
import time

# Your Space URL — HuggingFace converts repo name to this format
SPACE_URL = f'https://{YOUR_USERNAME}-privaware-api.hf.space'

print(f'Waiting for Space to come online at: {SPACE_URL}')
print('Checking every 30 seconds (may take up to 7 minutes)...')

for attempt in range(15):
    try:
        response = requests.get(f'{SPACE_URL}/health', timeout=10)
        if response.status_code == 200:
            print(f'\nSpace is LIVE after {(attempt+1)*30} seconds!')
            print(f'Response: {response.json()}')
            break
    except Exception:
        pass
    print(f'  Attempt {attempt+1}/15 — not ready yet, waiting 30s...')
    time.sleep(30)
else:
    print('\nSpace is taking longer than expected.')
    print(f'Check manually at: https://huggingface.co/spaces/{SPACE_REPO}')
    print('Once it shows Running status, run Cell 10.')

## Cell 10 — Run a full end-to-end test against your live API

In [ ]:
import requests

SPACE_URL = f'https://{YOUR_USERNAME}-privaware-api.hf.space'

print('Testing your LIVE deployed API...')
print(f'API URL: {SPACE_URL}')
print('=' * 55)

# Health check
r = requests.get(f'{SPACE_URL}/health')
print(f'Health check: {r.json()}')

# Test URLs
test_cases = [
    'https://www.google.com',
    'https://github.com',
    'http://paypal-secure-login.xyz/verify',
    'http://amazon-suspended-account.tk/restore',
]

print('\nLive URL scan results:')
print('-' * 55)
for url in test_cases:
    r = requests.post(f'{SPACE_URL}/scan-url', json={'url': url})
    data = r.json()
    flag = '🚨' if data['is_phishing'] else '✅'
    print(f"{flag} {data['label']} | Risk: {data['risk_score']}/100 | {url[:50]}")

print('\n' + '='*55)
print('DAY 3 COMPLETE!')
print(f'Your model is live at: https://huggingface.co/{REPO_ID}')
print(f'Your API is live at:   {SPACE_URL}')
print('Save both URLs — you need them for the extension in Day 5.')
print('Message me for Day 4 (privacy policy model).')
print('='*55)